In [3]:
import torch
import torch.nn as nn
class TransformerEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len):
        super().__init__()

        self.token_emb = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_model)

        self.pos_emb = nn.Embedding(num_embeddings=max_len, embedding_dim=d_model)

In [5]:
import torch.nn.functional as f
#self attention layer

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, max_len):
        super().__init__()

        self.c_attn = nn.Linear(d_model, 3 * d_model)
        self.c_proj = nn.Linear(d_model, d_model)

        self.register_buffer("mask", torch.tril(torch.ones(max_len, max_len)))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(C, dim=-1)


        scores = (q @ k.transpose(-2, -1)) * (1.0 / (C ** 0.5))


        scores = scores.masked_fill(self.mask[:T, :T] == 0, float('-inf'))


        probs = f.softmax(scores, dim=-1)


        out = probs @ v
        return self.c_proj(out)

In [6]:
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model)
        )

    def forward(self, x):
        return self.net(x)

In [7]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, max_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, max_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [10]:
import torch
import torch.nn as nn

class model(nn.Module):
    def __init__(self, vocab_size, d_model, max_len, n_layer=4):
        super().__init__()
        self.wordemb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, max_len) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx):
        B, T = idx.shape

        pos = torch.arange(0, T, device=idx.device)

        x = self.wordemb(idx) + self.pos_emb(pos)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        return logits